# 🩸 Blood Donation Analytics & Donor Intelligence Platform
## End-to-End EDA Notebook
**Tech Stack:** Python · Pandas · NumPy · Matplotlib · Seaborn

---
### Sections
1. Setup & Data Loading
2. Data Quality Report
3. Donor Analytics
4. Blood Request Analysis
5. Donation Trends
6. Supply vs Demand
7. Key Insights & KPIs


In [ ]:
# ── Cell 1: Imports & config ──────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings, os

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (10, 5)})

CLEANED = '../data/cleaned/'
print('Libraries loaded ✓')

In [ ]:
# ── Cell 2: Load cleaned datasets ─────────────────────────────
donors    = pd.read_csv(CLEANED + 'donors_clean.csv',           parse_dates=['last_donation_date'])
requests  = pd.read_csv(CLEANED + 'blood_requests_clean.csv',   parse_dates=['request_date'])
donations = pd.read_csv(CLEANED + 'donation_records_clean.csv', parse_dates=['donation_date'])

print(f'Donors    : {len(donors):,} rows  |  {donors.shape[1]} columns')
print(f'Requests  : {len(requests):,} rows  |  {requests.shape[1]} columns')
print(f'Donations : {len(donations):,} rows  |  {donations.shape[1]} columns')

In [ ]:
# ── Cell 3: Data quality report ───────────────────────────────
for name, df in [('Donors', donors), ('Requests', requests), ('Donations', donations)]:
    print(f'\n── {name} ──')
    print(f'  Shape     : {df.shape}')
    print(f'  Nulls     :\n{df.isnull().sum()[df.isnull().sum()>0].to_string()}')
    print(f'  Duplicates: {df.duplicated().sum()}')

In [ ]:
# ── Cell 4: Blood group distribution (donors) ─────────────────
bg_counts = donors['blood_group'].value_counts()
colors = sns.color_palette('Set2', len(bg_counts))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bg_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('Donor Count by Blood Group', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Blood Group'); axes[0].set_ylabel('Donors')
axes[0].tick_params(axis='x', rotation=0)

axes[1].pie(bg_counts, labels=bg_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=140, pctdistance=0.82)
axes[1].set_title('Blood Group Share (%)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Cell 5: Donor age distribution ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(donors['age'].dropna(), bins=20, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(donors['age'].mean(), color='red', linestyle='--', label=f"Mean: {donors['age'].mean():.1f}")
axes[0].axvline(donors['age'].median(), color='orange', linestyle='--', label=f"Median: {donors['age'].median():.1f}")
axes[0].set_title('Age Distribution of Donors', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Age'); axes[0].set_ylabel('Count'); axes[0].legend()

bins   = [17, 25, 35, 45, 55, 65]
labels = ['18-25','26-35','36-45','46-55','56-65']
donors['age_group'] = pd.cut(donors['age'], bins=bins, labels=labels)
ag = donors['age_group'].value_counts().sort_index()
ag.plot(kind='bar', ax=axes[1], color=sns.color_palette('muted', len(ag)), edgecolor='white')
axes[1].set_title('Donors by Age Group', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Age Group'); axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
# ── Cell 6: Availability by blood group ──────────────────────
avail = donors.groupby(['blood_group','availability_status']).size().unstack(fill_value=0)
avail.plot(kind='bar', stacked=True, figsize=(10,5),
           color=['#2ecc71','#e74c3c'], edgecolor='white')
plt.title('Donor Availability by Blood Group', fontsize=14, fontweight='bold')
plt.xlabel('Blood Group'); plt.ylabel('Donors')
plt.legend(title='Status'); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
# ── Cell 7: Top 15 donor cities ──────────────────────────────
top_cities = donors['city'].value_counts().head(15)
top_cities.sort_values().plot(kind='barh', figsize=(10,6), color='steelblue', edgecolor='white')
plt.title('Top 15 Cities by Donor Count', fontsize=14, fontweight='bold')
plt.xlabel('Number of Donors'); plt.tight_layout(); plt.show()

In [ ]:
# ── Cell 8: Blood demand – most requested blood groups ────────
demand = requests.groupby('blood_group')['units_required'].sum().sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

demand.plot(kind='bar', ax=axes[0], color=sns.color_palette('Reds_r', len(demand)), edgecolor='white')
axes[0].set_title('Total Units Demanded by Blood Group', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Blood Group'); axes[0].set_ylabel('Units Required')
axes[0].tick_params(axis='x', rotation=0)

urg = requests['urgency_level'].value_counts()
axes[1].pie(urg, labels=urg.index, autopct='%1.1f%%',
            colors=['#e74c3c','#e67e22','#3498db','#2ecc71'], startangle=140)
axes[1].set_title('Requests by Urgency Level', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Cell 9: Monthly request trend ────────────────────────────
requests['month'] = requests['request_date'].dt.to_period('M')
monthly_req = requests.groupby('month').agg(
    total_requests=('request_id','count'),
    total_units=('units_required','sum')
).reset_index()
monthly_req['month_dt'] = monthly_req['month'].dt.to_timestamp()

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()
ax1.bar(monthly_req['month_dt'], monthly_req['total_requests'], color='#3498db', alpha=0.7, label='Requests')
ax2.plot(monthly_req['month_dt'], monthly_req['total_units'], color='#e74c3c', linewidth=2, marker='o', markersize=4, label='Units')
ax1.set_xlabel('Month'); ax1.set_ylabel('Request Count', color='#3498db')
ax2.set_ylabel('Units Required', color='#e74c3c')
plt.title('Monthly Blood Request Trend', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Cell 10: Monthly donation trend ──────────────────────────
donations['month'] = donations['donation_date'].dt.to_period('M')
monthly_don = donations.groupby('month').agg(
    total_donations=('donation_id','count'),
    total_units=('units_donated','sum')
).reset_index()
monthly_don['month_dt'] = monthly_don['month'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(monthly_don['month_dt'], monthly_don['total_units'], alpha=0.3, color='green')
ax.plot(monthly_don['month_dt'], monthly_don['total_units'], color='green', linewidth=2)
ax.set_title('Monthly Donation Volume (Units)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month'); ax.set_ylabel('Units Donated')
plt.tight_layout(); plt.show()

In [ ]:
# ── Cell 11: Supply vs Demand comparison ─────────────────────
supply = donations.groupby('blood_group')['units_donated'].sum().rename('Supply')
demand = requests.groupby('blood_group')['units_required'].sum().rename('Demand')
svd = pd.concat([supply, demand], axis=1).fillna(0).astype(int)
svd['Deficit'] = svd['Demand'] - svd['Supply']
svd = svd.sort_values('Deficit', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
svd[['Supply','Demand']].plot(kind='bar', ax=axes[0],
    color=['#2ecc71','#e74c3c'], edgecolor='white')
axes[0].set_title('Supply vs Demand by Blood Group', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Blood Group'); axes[0].set_ylabel('Units')
axes[0].tick_params(axis='x', rotation=0)

bar_colors = ['#e74c3c' if x > 0 else '#2ecc71' for x in svd['Deficit']]
svd['Deficit'].plot(kind='bar', ax=axes[1], color=bar_colors, edgecolor='white')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Net Deficit (Demand − Supply)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Blood Group'); axes[1].set_ylabel('Units (negative = surplus)')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

print('\nSupply vs Demand Table:')
print(svd.to_string())

In [ ]:
# ── Cell 12: Gender breakdown of donors ──────────────────────
gender = donors['gender'].value_counts()
gender.plot(kind='pie', autopct='%1.1f%%',
            colors=['#3498db','#e91e8c','#95a5a6'],
            startangle=90, figsize=(6,6))
plt.title('Donor Gender Distribution', fontsize=14, fontweight='bold')
plt.ylabel('')
plt.tight_layout(); plt.show()

In [ ]:
# ── Cell 13: Key KPI Summary ─────────────────────────────────
kpis = {
    'Total Donors'             : len(donors),
    'Available Donors'         : (donors['availability_status']=='Available').sum(),
    'Average Donor Age'        : round(donors['age'].mean(), 1),
    'Total Blood Requests'     : len(requests),
    'Total Units Requested'    : requests['units_required'].sum(),
    'Critical Requests'        : (requests['urgency_level']=='Critical').sum(),
    'Total Donation Events'    : len(donations),
    'Total Units Donated'      : donations['units_donated'].sum(),
    'Most Demanded Blood Group': requests.groupby('blood_group')['units_required'].sum().idxmax(),
    'Most Available Blood Group': donors[donors['availability_status']=='Available']['blood_group'].value_counts().idxmax(),
}
print('\n══════════════ KEY KPIs ══════════════')
for k, v in kpis.items():
    print(f'  {k:<30} : {v}')